In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
# Esto mostrará el contenido de la carpeta donde debería estar tu Drive
print(os.listdir('/content/drive/MyDrive/Tazas'))

['dataset_tazas', 'dataset_tazas.zip']


In [8]:
import os

# 1. Listemos paso a paso para ver dónde se "pierde" la ruta
base = '/content/drive/MyDrive/Tazas'
print(f"Contenido de {base}: {os.listdir(base)}")

ruta_dataset = os.path.join(base, 'dataset_tazas')
print(f"Contenido de {ruta_dataset}: {os.listdir(ruta_dataset)}")

ruta_train = os.path.join(ruta_dataset, 'train')
print(f"¿Existe 'train' en {ruta_dataset}?: {os.path.exists(ruta_train)}")

if os.path.exists(ruta_train):
    print("Contenido de 'train':", os.listdir(ruta_train))

Contenido de /content/drive/MyDrive/Tazas: ['dataset_tazas', 'dataset_tazas.zip']
Contenido de /content/drive/MyDrive/Tazas/dataset_tazas: ['test', 'train']
¿Existe 'train' en /content/drive/MyDrive/Tazas/dataset_tazas?: True
Contenido de 'train': ['no_taza', 'taza']


In [6]:
ruta_prueba = '/content/drive/MyDrive/Tazas/dataset_tazas/train'
if os.path.exists(ruta_prueba):
    print("¡Ruta encontrada!")
    print("Contenido:", os.listdir(ruta_prueba))
else:
    print("Error: La ruta sigue sin encontrarse. Verifica mayúsculas y nombres.")

¡Ruta encontrada!
Contenido: ['no_taza', 'taza']


In [7]:
import shutil
import os

# Borrar archivos ocultos que suelen causar errores de lectura
def limpiar_drive(path):
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.startswith('.'): # Borra archivos tipo .DS_Store o ocultos
                os.remove(os.path.join(root, f))

limpiar_drive('/content/drive/MyDrive/Tazas/dataset_tazas/')
print("Limpieza completada.")

Limpieza completada.


In [10]:
import tensorflow as tf
import os # Added this line

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

ruta_base_local = '/content/data_local/dataset_tazas'

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(ruta_base_local, 'train'),
    image_size=(224, 224),
    batch_size=32,
    label_mode='binary'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(ruta_base_local, 'test'),
    image_size=(224, 224),
    batch_size=32,
    label_mode='binary'
)

# Optimización para lectura rápida
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

FileNotFoundError: [Errno 2] No such file or directory: '/content/data_local/dataset_tazas/train'

Celda A: Definición del Modelo - Reconstruye tu modelo

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Definir la arquitectura de nuevo
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid', name="confidence_score")
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("✅ Modelo reconstruido correctamente.")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Modelo reconstruido correctamente.


Llevar a Memoria Local

In [14]:
# 1. Copia tu ZIP desde Drive al entorno local de Colab
!cp /content/drive/MyDrive/Tazas/dataset_tazas.zip /content/dataset_tazas.zip

# 2. Asegurarse de que la carpeta de destino esté limpia antes de descomprimir
import shutil
import os
if os.path.exists('/content/data_local'):
    shutil.rmtree('/content/data_local')
    print("Carpeta '/content/data_local' eliminada para una extracción limpia.")

# 3. Descomprime en la carpeta local de Colab (esto es rapidísimo)
!unzip -q /content/dataset_tazas.zip -d /content/data_local

# 4. Ajusta tu código para que lea desde '/content/data_local/train' en lugar de Drive

Carpeta '/content/data_local' eliminada para una extracción limpia.


In [12]:
# Listar qué hay dentro de la carpeta local donde descomprimiste
import os
ruta_local = '/content/data_local/dataset_tazas' # O la ruta que hayas usado en tu !unzip
print(os.listdir(ruta_local))
print(os.listdir(os.path.join(ruta_local, 'train')))

# Verifica si dentro está la carpeta 'train'
print(os.listdir(os.path.join(ruta_local, 'train')))

['train', 'test']
['no_taza', 'taza']
['no_taza', 'taza']


Celda B: El Entrenamiento (la IA "estudia")

In [17]:
!pip install -U tf2onnx -q
import tf2onnx
import tensorflow as tf
import os
import shutil # Import shutil for directory operations
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Redefine the model here to ensure it's always available
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid', name="confidence_score")
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("✅ Modelo reconstruido correctamente en esta celda.")

# --- Start of Dataset Loading (moved from m3BjCBb5FPiU) ---
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
ruta_base_local = '/content/data_local/dataset_tazas'

train_ds = None # Initialize to None
test_ds = None # Initialize to None

print(f"DEBUG: Intentando cargar train_ds desde: {os.path.join(ruta_base_local, 'train')}")
try:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(ruta_base_local, 'train'),
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='binary'
    )
    print("DEBUG: train_ds cargado exitosamente.")
except Exception as e:
    print(f"ERROR al cargar train_ds: {e}")

print(f"DEBUG: Intentando cargar test_ds desde: {os.path.join(ruta_base_local, 'test')}")
try:
    test_ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(ruta_base_local, 'test'),
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='binary'
    )
    print("DEBUG: test_ds cargado exitosamente.")
except Exception as e:
    print(f"ERROR al cargar test_ds: {e}")

# Check if datasets were loaded successfully before prefetching
if train_ds and test_ds:
    # Optimización para lectura rápida
    train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    print("✅ Conjuntos de datos (train_ds, test_ds) cargados y optimizados correctamente en esta celda.")
else:
    print("❌ No se pudieron cargar uno o ambos conjuntos de datos. Deteniendo ejecución.")
    # Raise an error if datasets are not available to stop further execution
    raise ValueError("Datasets train_ds or test_ds could not be loaded.")

# --- End of Dataset Loading ---

# Asegúrate de crear la carpeta antes
if not os.path.exists("entrenamiento_tazas"):
    os.makedirs("entrenamiento_tazas")

checkpoint_path = "entrenamiento_tazas/cp-{epoch:04d}.weights.h5" # <--- CAMBIO AQUÍ
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    verbose=1
)


✅ Modelo reconstruido correctamente en esta celda.
DEBUG: Intentando cargar train_ds desde: /content/data_local/dataset_tazas/train
Found 7107 files belonging to 2 classes.
DEBUG: train_ds cargado exitosamente.
DEBUG: Intentando cargar test_ds desde: /content/data_local/dataset_tazas/test
Found 1839 files belonging to 2 classes.
DEBUG: test_ds cargado exitosamente.
✅ Conjuntos de datos (train_ds, test_ds) cargados y optimizados correctamente en esta celda.
Epoch 1/10
223/223 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9069 - loss: 0.2357
Epoch 1: saving model to entrenamiento_tazas/cp-0001.weights.h5

Epoch 1: finished saving model to entrenamiento_tazas/cp-0001.weights.h5
223/223 ━━━━━━━━━━━━━━━━━━━━ 386s 2s/step - accuracy: 0.9446 - loss: 0.1539 - val_accuracy: 0.9391 - val_loss: 0.1857
Epoch 2/10
223/223 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9632 - loss: 0.0969
Epoch 2: saving model to entrenamiento_tazas/cp-0002.weights.h5

Epoch 2: finished saving model to entrenamiento_

ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=saved_model_tazas.

Celda C: Exportar a .ONNX

In [18]:
for layer in model.layers:
    print(f"Capa: {layer.name}")

Capa: mobilenetv2_1.00_224
Capa: global_average_pooling2d_2
Capa: confidence_score


In [19]:
# Cargar el último checkpoint guardado
model.load_weights('entrenamiento_tazas/cp-0010.weights.h5') # Asegúrate que este nombre exista
print("✅ Pesos cargados correctamente.")

✅ Pesos cargados correctamente.


In [24]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, Input

# 1. Definición clara
inputs = Input(shape=(224, 224, 3), name="cam_input")
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')(inputs)
pooling = layers.GlobalAveragePooling2D()(base_model)
# Definimos la salida con nombre y la conectamos explícitamente
outputs = layers.Dense(1, activation='sigmoid', name="confidence_score")(pooling)

model = models.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("✅ Modelo reconstruido con entrada y salida explícitas.")

# 2. Convertir inmediatamente (Sin necesidad de entrenar de nuevo,
# solo estamos preparando la estructura para exportar)
import tf2onnx

spec = (tf.TensorSpec((None, 224, 224, 3), tf.float32, name="cam_input"),)

# Convertir
model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=spec,
    opset=12,
    output_path="modelo_tazas.onnx"
)

print("✅ ¡Archivo generado exitosamente!")
from google.colab import files
files.download("modelo_tazas.onnx")

✅ Modelo reconstruido con entrada y salida explícitas.
✅ ¡Archivo generado exitosamente!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>